# Insurance Risk Analytics

This notebook explores and analyzes insurance risk data.

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## Load and Explore Data

In [ ]:
# Load your insurance data here
# df = pd.read_csv('insurance_data.csv')
# df.head()

## Data Cleaning

In [ ]:
# Load insurance data
df = pd.read_csv('insurance_data.csv')

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types and missing values:")
print(df.info())

# Clean data - drop missing values
df = df.dropna()
print(f"\nAfter cleaning - Dataset shape: {df.shape}")

# Create data directory if it doesn't exist
import os
os.makedirs('../data', exist_ok=True)

# Save cleaned data
df.to_csv("../data/insurance_data_cleaned.csv", index=False)
print("\nCleaned data saved to ../data/insurance_data_cleaned.csv")

## Data Analysis and Visualization

In [ ]:
# Statistical Summary
print("=== Statistical Summary ===")
print(df.describe())

print("\n=== Average Charges by Smoker Status ===")
print(df.groupby('smoker')['charges'].mean())

print("\n=== Claim Probability by Region ===")
print(df.groupby('region')['claim_probability'].mean())

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of charges
axes[0, 0].hist(df['charges'], bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Charges')
axes[0, 0].set_xlabel('Charges')
axes[0, 0].set_ylabel('Frequency')

# Box plot: Charges by Smoker
df.boxplot(column='charges', by='smoker', ax=axes[0, 1])
axes[0, 1].set_title('Charges by Smoking Status')
axes[0, 1].set_ylabel('Charges')

# Scatter: BMI vs Charges
axes[1, 0].scatter(df['bmi'], df['charges'], alpha=0.5, color='green')
axes[1, 0].set_title('BMI vs Charges')
axes[1, 0].set_xlabel('BMI')
axes[1, 0].set_ylabel('Charges')

# Bar plot: Average charges by region
df.groupby('region')['charges'].mean().plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Average Charges by Region')
axes[1, 1].set_ylabel('Average Charges')
axes[1, 1].set_xlabel('Region')

plt.tight_layout()
plt.show()

print("Visualizations complete!")

In [ ]:
# Correlation Analysis
print("=== Correlation Matrix ===")
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = df[numeric_cols].corr()
print(correlation_matrix)

# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Insurance Data')
plt.tight_layout()
plt.show()

## Risk Modeling

In [ ]:
# Prepare data for modeling
from sklearn.preprocessing import LabelEncoder

# Create a copy for modeling
df_model = df.copy()

# Encode categorical variables
le_dict = {}
for col in ['gender', 'smoker', 'region']:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

# Split features and target
X = df_model.drop('claim_probability', axis=1)
y = df_model['claim_probability']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Train Random Forest Model
print("\n=== Training Random Forest Model ===")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train, y_train)

# Evaluate model
train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== Feature Importance ===")
print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance - Insurance Risk Prediction')
plt.tight_layout()
plt.show()

## Data Versioning with DVC

In [ ]:
# To push data to DVC remote storage, uncomment and run:
# import subprocess
# subprocess.run(['dvc', 'push'], check=True)
# print("Data pushed to DVC remote storage successfully!")

# To pull data from DVC remote storage, uncomment and run:
# subprocess.run(['dvc', 'pull'], check=True)
# print("Data pulled from DVC remote storage successfully!")